# Analyse: Schärfe


In [3]:
import cv2
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

DATA_DIR = Path('../../data')
COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '12_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_visual_sharpness.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'
DEFAULT_MAX_FRAMES_PER_VIDEO = 60
SOURCE_FILTER = None

df = pd.read_csv(COMBINED_CSV)
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'
df[video_id_col] = df[video_id_col].astype(str)

def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()

video_ids = df[video_id_col].astype(str)
df['has_frames'] = [has_frames(video_id) for video_id in video_ids]
missing_ids = video_ids[~df['has_frames']].tolist()

print(f'Loaded {len(df)} rows from {COMBINED_CSV.name}')


Loaded 500 rows from 01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv


In [4]:
METRIC_COLUMNS = [
    'sharpness_laplacian_mean',
    'sharpness_laplacian_std',
    'processed_frame_count',
]

print(f"Videos with frame folder: {df['has_frames'].sum()}")
print(f'Videos missing frame folder: {len(missing_ids)}')
if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

# Schärfe nur für Videos mit extrahierten Frames berechnen
df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')

def get_duration_seconds(row: pd.Series):
    for col in ('video_duration_seconds', 'duration_seconds', 'duration', 'video_duration'):
        if col in row.index:
            value = row.get(col, np.nan)
            if pd.notna(value):
                return value
    return np.nan

def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []
    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.jpg'))
    if not frame_files:
        return []

    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, len(frame_files)))
    indices = np.linspace(0, len(frame_files) - 1, num=target_frames, dtype=int)
    return [frame_files[idx] for idx in np.unique(indices)]

def compute_laplacian_variance(image_path: Path) -> float:
    try:
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            return np.nan
        return float(cv2.Laplacian(image, cv2.CV_64F).var())
    except Exception as exc:
        print(f'Error reading {image_path}: {exc}')
        return np.nan

def extract_visual_sharpness_metrics(row: pd.Series) -> dict:
    video_id = row[video_id_col]
    duration_seconds = get_duration_seconds(row)
    folder = FRAME_ROOT / video_id
    available_frame_paths = sorted(folder.glob('*.jpeg')) if folder.is_dir() else []
    if not available_frame_paths and folder.is_dir():
        available_frame_paths = sorted(folder.glob('*.jpg'))

    frame_paths = sample_frame_paths(video_id, duration_seconds=duration_seconds)
    sharpness_values = [compute_laplacian_variance(frame_path) for frame_path in frame_paths]
    sharpness_values = [value for value in sharpness_values if pd.notna(value)]

    metrics = {'video_id': video_id}
    for col in METRIC_COLUMNS:
        metrics[col] = np.nan

    metrics['processed_frame_count'] = len(sharpness_values)

    if sharpness_values:
        values = np.asarray(sharpness_values, dtype=float)
        metrics['sharpness_laplacian_mean'] = float(values.mean())
        metrics['sharpness_laplacian_std'] = float(values.std())

    return metrics

result_rows = [extract_visual_sharpness_metrics(row) for _, row in tqdm(df.iterrows(), total=len(df))]
feature_df = pd.DataFrame(result_rows)
merged = df.merge(feature_df, on='video_id', how='left') if 'video_id' in df.columns else df.merge(feature_df, left_on=video_id_col, right_on='video_id', how='left')
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT_CSV, index=False)
print(merged[[col for col in ['influencer_type'] + METRIC_COLUMNS if col in merged.columns]].describe(include='all'))
print(f'Wrote {OUTPUT_CSV} with shape {merged.shape}')
merged.head()


Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


100%|██████████| 500/500 [01:38<00:00,  5.06it/s]


       influencer_type  sharpness_laplacian_mean  sharpness_laplacian_std  \
count              500                500.000000               500.000000   
unique               2                       NaN                      NaN   
top                 ai                       NaN                      NaN   
freq               250                       NaN                      NaN   
mean               NaN                565.464822               171.697673   
std                NaN                453.992090               219.558396   
min                NaN                 44.929979                 4.924166   
25%                NaN                280.499375                53.857238   
50%                NaN                446.880964               107.541760   
75%                NaN                683.306609               209.508362   
max                NaN               3327.960330              2522.630720   

        processed_frame_count  
count              500.000000  
unique     

,video_id,video_thread_id,author_username,author_displayname,author_follower_count,author_like_count_total,author_video_count,author_avatar_url,video_caption,video_stickers,...,video_has_video,video_duration_seconds,influencer,influencer_clean,video_engagement_rate,eng_bin,has_frames,sharpness_laplacian_mean,sharpness_laplacian_std,processed_frame_count
0,7516243181650988334,7516243181650988334,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,oop 🫢🫢 #fyp #trending #googleveo3 #ai #viral ...,🫢,...,True,6.958,ai.kalai,ai.kalai,0.011906,Low,True,391.376452,71.226256,7
1,7515925552549678378,7515925552549678378,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,"and if i’m fake, i ain’t notice cause my money...",NaN,...,True,9.500,ai.kalai,ai.kalai,0.013038,Low,True,178.094112,54.099509,10
2,7521159314757684535,7521159314757684535,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,uh oh!! #peanutbutter #groceryshopping #grocer...,teaching yall how i steal cause why this almon...,...,True,8.000,ai.kalai,ai.kalai,0.006939,Low,True,708.159818,77.394082,8
3,7521486299098778894,7521486299098778894,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,insanee #movies #movienight #popcorn #fastfood...,NaN,...,True,7.300,ai.kalai,ai.kalai,0.010583,Low,True,807.677676,424.034322,8
4,7521490969468865847,7521490969468865847,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,and he knows that 🎬 #acting #actor #actress #v...,NaN,...,True,7.967,ai.kalai,ai.kalai,0.009967,Low,True,399.199029,37.780564,8
